# Data Preprocessing & Feature Engineering

## Objective

Prepare the dataset for machine learning by cleaning the data, handling missing values, creating useful features, encoding categorical variables, scaling numerical features, and building a preprocessing pipeline.

# Import Libraries

In [75]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    OrdinalEncoder
)

pd.set_option("display.max_columns", None)

# Load Dataset

Load the cleaned dataset from the previous EDA stage.

In [76]:
df = pd.read_csv("data/train-test.csv")

df.head()

,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


# Dataset Overview

Verify the dataset before preprocessing.

In [77]:
df.shape

(48000, 14)

In [78]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  str    
 1   pickup        48000 non-null  str    
 2   delivery      48000 non-null  str    
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  str    
 9   weight        47700 non-null  float64
 10  date          48000 non-null  str    
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), str(5)
memory usage: 5.1 MB


# Handle Missing Values

## Objective

Identify missing values before choosing an imputation strategy.

In [79]:
missing = df.isnull().sum()

missing = missing[missing > 0]

missing.sort_values(ascending=False)

market_index    374
weight          300
dtype: int64

# Handle Invalid Values

Look for impossible or incorrect values.

In [80]:
(df['weight']<0).sum()

np.int64(292)

In [81]:
df.describe()

,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,market_index,quote_signal,posted_rate
count,48000.000000,48000.000000,48000.000000,48000.000000,48000.000000,47700.000000,47626.000000,48000.000000,48000.000000
mean,35.647545,-90.928964,35.641175,-90.857310,1135.856654,31028.844004,1.083387,2.062468,2373.980682
std,4.315285,13.482431,4.317199,13.476589,728.564416,9391.440620,0.168091,0.291391,1486.493245
min,28.357650,-121.698490,28.357650,-121.698490,70.000000,-47500.000000,0.676390,0.692280,57.220000
25%,31.986910,-98.400590,31.986910,-98.400590,550.400000,25800.000000,0.949670,1.891030,1251.555000
50%,35.294790,-88.089150,35.294790,-87.528710,953.300000,31436.500000,1.055800,2.055750,2030.760000
75%,39.411040,-83.285060,39.411040,-83.285060,1645.525000,37018.000000,1.219590,2.221685,3330.750000
max,44.302960,-69.500000,44.302960,-69.500000,3439.800000,47500.000000,1.467780,3.610350,25533.000000


In [82]:
df[df['weight']<0].describe()

,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,market_index,quote_signal,posted_rate
count,292.000000,292.000000,292.000000,292.000000,292.000000,292.000000,289.000000,292.000000,292.000000
mean,35.493040,-91.581744,35.706866,-90.117659,1120.920205,-31724.195205,1.096322,2.041966,2389.786233
std,4.070043,13.255564,4.508908,13.068713,733.132047,8262.536326,0.171562,0.298395,1641.761285
min,28.357650,-121.698490,28.357650,-121.698490,75.100000,-47500.000000,0.789880,1.157730,247.370000
25%,32.546280,-96.521915,31.823792,-95.895690,531.975000,-37284.250000,0.961550,1.868555,1257.410000
50%,35.294790,-88.251950,35.373760,-88.170550,930.450000,-31821.500000,1.065710,2.033965,1994.960000
75%,39.223170,-84.045170,39.445375,-82.783790,1537.125000,-25928.500000,1.241320,2.194368,3185.257500
max,44.302960,-69.500000,44.302960,-69.500000,3161.700000,-5000.000000,1.431890,3.312330,14561.210000


In [89]:
df.loc[df['weight'] < 0, 'weight'] = np.nan

In [90]:
df['weight'].isnull().sum()

np.int64(592)

# Convert Data Types

Convert columns into their correct data types.

In [91]:
df['date']=pd.to_datetime(df['date'])

# Feature Engineering

Extract useful information from existing features.

In [92]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)

# Remove Unnecessary Columns

Drop columns that should not be used for training.

In [93]:
df.drop(columns=['load_id','date'],inplace=True)

In [94]:
df.columns

Index(['pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat',
       'delivery_lon', 'distance', 'equipment', 'weight', 'market_index',
       'quote_signal', 'posted_rate', 'year', 'month', 'day', 'day_of_week',
       'week_of_year'],
      dtype='str')

# Separate Features and Target

In [95]:
X = df.drop(columns=["posted_rate"])

y = df["posted_rate"]

# Identify Numerical and Categorical Features

In [96]:
categorical_features = X.select_dtypes(include="object").columns

numerical_features = X.select_dtypes(exclude="object").columns

print(categorical_features)

print(numerical_features)

Index(['pickup', 'delivery', 'equipment'], dtype='str')
Index(['pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance',
       'weight', 'market_index', 'quote_signal', 'year', 'month', 'day',
       'day_of_week', 'week_of_year'],
      dtype='str')


/var/folders/4c/5h0nzz294t390b8nn0gqgtl80000gn/T/ipykernel_99221/2735852008.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include="object").columns


##  preprocessing without scalling



In [97]:
preprocessor_not_scaled = ColumnTransformer(
    transformers=[
        ("num",SimpleImputer(strategy="median"),numerical_features,),
        ("cat",Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))]),
            categorical_features)])

## preprocessing with scalling

In [98]:
preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("num",Pipeline([
            ('imputer',SimpleImputer(strategy='median')),
            ('scale',StandardScaler()) ]),
            numerical_features
         ),
        ("cat",Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))]),
            categorical_features)])

## train and test spliting


In [100]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

## using  transform

In [101]:
X_train_not_scaled_transformed=preprocessor_not_scaled.fit_transform(X_train)
X_test_not_scaled_transformed=preprocessor_not_scaled.transform(X_test)


In [102]:
X_train_scaled_transformed=preprocessor_scaled.fit_transform(X_train)
X_test_scaled_transformed=preprocessor_scaled.transform(X_test)

In [103]:
X_test_not_scaled_transformed=X_test_not_scaled_transformed.toarray()
X_train_not_scaled_transformed=X_train_not_scaled_transformed.toarray()

In [104]:
X_test_scaled_transformed=X_test_scaled_transformed.toarray()
X_train_scaled_transformed=X_train_scaled_transformed.toarray()

In [105]:
features_name_1=preprocessor_not_scaled.get_feature_names_out()
X_train_not_scaled_transformed=pd.DataFrame(X_train_not_scaled_transformed,columns=features_name_1)
X_test_not_scaled_transformed=pd.DataFrame(X_test_not_scaled_transformed,columns=features_name_1)

In [106]:
X_test_not_scaled_transformed

,num__pickup_lat,num__pickup_lon,num__delivery_lat,num__delivery_lon,num__distance,num__weight,num__market_index,num__quote_signal,num__year,num__month,num__day,num__day_of_week,num__week_of_year,cat__pickup_Albany,cat__pickup_Albuquerque,cat__pickup_Amarillo,cat__pickup_Atlanta,cat__pickup_Austin,cat__pickup_Bakersfield,cat__pickup_Baltimore,cat__pickup_Baton Rouge,cat__pickup_Birmingham,cat__pickup_Boston,cat__pickup_Buffalo,cat__pickup_Charleston,cat__pickup_Chattanooga,cat__pickup_Cincinnati,cat__pickup_Columbia,cat__pickup_Corpus Christi,cat__pickup_Dallas,cat__pickup_Dayton,cat__pickup_Detroit,cat__pickup_El Paso,cat__pickup_Fort Wayne,cat__pickup_Fresno,cat__pickup_Grand Rapids,cat__pickup_Green Bay,cat__pickup_Greensboro,cat__pickup_Harrisburg,cat__pickup_Hartford,cat__pickup_Houston,cat__pickup_Indianapolis,cat__pickup_Jacksonville,cat__pickup_Kansas City,cat__pickup_Las Vegas,cat__pickup_Lexington,cat__pickup_Little Rock,cat__pickup_Los Angeles,cat__pickup_Louisville,cat__pickup_Lubbock,cat__pickup_Madison,cat__pickup_Memphis,cat__pickup_Milwaukee,cat__pickup_Mobile,cat__pickup_Montgomery,cat__pickup_Nashville,cat__pickup_New Orleans,cat__pickup_New York,cat__pickup_Oklahoma City,cat__pickup_Philadelphia,cat__pickup_Phoenix,cat__pickup_Providence,cat__pickup_Raleigh,cat__pickup_Reno,cat__pickup_Richmond,cat__pickup_Salt Lake City,cat__pickup_San Antonio,cat__pickup_San Francisco,cat__pickup_Savannah,cat__pickup_Shreveport,cat__pickup_St. Louis,cat__pickup_Syracuse,cat__pickup_Tampa,cat__pickup_Toledo,cat__pickup_Tucson,cat__pickup_Tulsa,cat__pickup_Washington,cat__delivery_Albany,cat__delivery_Albuquerque,cat__delivery_Amarillo,cat__delivery_Atlanta,cat__delivery_Austin,cat__delivery_Bakersfield,cat__delivery_Baltimore,cat__delivery_Baton Rouge,cat__delivery_Birmingham,cat__delivery_Boston,cat__delivery_Buffalo,cat__delivery_Charleston,cat__delivery_Chattanooga,cat__delivery_Cincinnati,cat__delivery_Columbia,cat__delivery_Corpus Christi,cat__delivery_Dallas,cat__delivery_Dayton,cat__delivery_Detroit,cat__delivery_El Paso,cat__delivery_Fort Wayne,cat__delivery_Fresno,cat__delivery_Grand Rapids,cat__delivery_Green Bay,cat__delivery_Greensboro,cat__delivery_Harrisburg,cat__delivery_Hartford,cat__delivery_Houston,cat__delivery_Indianapolis,cat__delivery_Jacksonville,cat__delivery_Kansas City,cat__delivery_Las Vegas,cat__delivery_Lexington,cat__delivery_Little Rock,cat__delivery_Los Angeles,cat__delivery_Louisville,cat__delivery_Lubbock,cat__delivery_Madison,cat__delivery_Memphis,cat__delivery_Milwaukee,cat__delivery_Mobile,cat__delivery_Montgomery,cat__delivery_Nashville,cat__delivery_New Orleans,cat__delivery_New York,cat__delivery_Oklahoma City,cat__delivery_Philadelphia,cat__delivery_Phoenix,cat__delivery_Providence,cat__delivery_Raleigh,cat__delivery_Reno,cat__delivery_Richmond,cat__delivery_Salt Lake City,cat__delivery_San Antonio,cat__delivery_San Francisco,cat__delivery_Savannah,cat__delivery_Shreveport,cat__delivery_St. Louis,cat__delivery_Syracuse,cat__delivery_Tampa,cat__delivery_Toledo,cat__delivery_Tucson,cat__delivery_Tulsa,cat__delivery_Washington,cat__equipment_Dry Van,cat__equipment_Flatbed,cat__equipment_Reefer
0,36.99152,-84.99876,31.80442,-88.25195,479.8,24292.0,0.99066,1.69328,2025.0,7.0,26.0,5.0,30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,43.43800,-86.29173,30.50134,-92.65374,1111.9,26504.0,0.80010,2.17300,2025.0,9.0,7.0,6.0,36.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.

In [107]:
features_name_2=preprocessor_scaled.get_feature_names_out()
X_train_scaled_transformed=pd.DataFrame(X_train_scaled_transformed,columns=features_name_2)
X_test_scaled_transformed=pd.DataFrame(X_test_scaled_transformed,columns=features_name_2)

In [108]:
import os

os.makedirs("optional", exist_ok=True)

In [109]:
X_train_scaled_transformed.to_csv("optional/X_train_scaled.csv", index=False)
X_test_scaled_transformed.to_csv("optional/X_test_scaled.csv", index=False)

X_train_not_scaled_transformed.to_csv("optional/X_train_not_scaled.csv", index=False)
X_test_not_scaled_transformed.to_csv("optional/X_test_not_scaled.csv", index=False)

y_train.to_csv("optional/y_train.csv", index=False)
y_test.to_csv("optional/y_test.csv", index=False)